# Experiment 7 vs 7e - generated tokens and results
Compares the CoT configurations of experiment 7 (0-, e1-, r4-think) with the ephemeral er4-think of experiment 7e, in Cooking.
Instead of the execution time, which the parallel runs of 7e cannot measure (their times are wall-clock times of games sharing the GPU), the cost is the **number of tokens generated** by the model:
it does not depend on the GPU nor on the other games played at the same time, and most of the execution time is spent generating them.

Runs locally: it only needs the result pickles, the logs and the Qwen3 tokenizer (no GPU, no model).

# Variables

In [ ]:
import os
import re
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bootstrap

game = "cooking"
run_levels = range(1, 10)

# model -> (folder, seeds used for the scores, seeds with a log, used for the tokens)
configurations = {
    "0":   ("./Experiment 7",  range(1, 11), range(0)), # no self-evaluation and no logs: only the scores
    "e1":  ("./Experiment 7",  range(1, 11), range(1, 11)),
    "r4":  ("./Experiment 7",  range(1, 11), range(11, 21)), # only the logs of seeds 11-20 were kept
    "er4": ("./Experiment 7e", range(1, 11), range(1, 11)),
}

with open("../Games/max_scores_cooking.pickle", "rb") as f:
    max_scores_cooking = pickle.load(f)

# Results
Normalized final score of each game, on seeds 1-10 for every configuration (the same games).

In [ ]:
def load_scores(model, level):
    folder, seeds, _ = configurations[model]
    with open(f"{folder}/{model}think_{game}_level{level}.pickle", "rb") as f:
        results_all_seeds = pickle.load(f)
    results_all_seeds = [results_all_seeds[seed - 1] for seed in seeds if seed <= len(results_all_seeds)] # the pickles hold seeds 1, 2, ... in order
    return np.array([run[-1][1] for run in results_all_seeds], dtype=float) / max_scores_cooking[level - 1]

scores = {model: {level: load_scores(model, level) for level in run_levels} for model in configurations}

# Generated tokens
The logs contain the text of every answer of the model, so the generated tokens are counted by tokenizing it again:
- normal turn: the logged command, plus 5 tokens (the empty `<think></think>` block and the end of turn, which the log does not show);
- self-evaluation: the logged reasoning and command, plus 2 tokens;
- a self-evaluation that reached the limit of 10000 tokens is logged as `help` or `look` without a reasoning: it is counted as 10000 tokens.

The counts were checked against the logs of 7e that also contain the real number of generated tokens (the `SPEED` lines): they match exactly, except for the very rare normal turns that reached their limit of 100 tokens, which cannot be told apart from a normal `help`/`look` (1 in about 900 turns).

The tokens of 0-think are not counted (no logs): it only has normal turns, about 13 tokens each.
r4-think is counted on seeds 11-20, the only ones whose logs were kept: different games from the scores, but of the same levels.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B") # only the tokenizer is downloaded

MARKERS = ("GAME ++++", "AGENT ----", "SELF-EVALUATION: +-", "CONTEXT INITIALIZED", "SPEED:", "LOOP DETECTED")
OVERHEAD = {"normal": 5, "think": 2} # tokens generated but not written in the log
THINK_CAP = 10000 # max_new_tokens of a self-evaluation

def log_turns(path):
    """(kind, text) of every answer of the model in the log, kind is "normal" or "think"."""
    turns, kind, lines = [], None, []
    for line in open(path, encoding="utf-8").read().split("\n") + ["CONTEXT INITIALIZED"]:
        if line.startswith(MARKERS):
            if kind is not None:
                turns.append((kind, "\n".join(lines).strip("\n")))
            kind = "normal" if line.startswith("AGENT") else "think" if line.startswith("SELF-EVALUATION") else None
            lines = []
        else:
            lines.append(line)
    return turns

def count_tokens(path):
    """(generated tokens, steps, stopped because of a loop) of the game in the log."""
    total = 0
    turns = log_turns(path)
    for kind, text in turns:
        if kind == "think" and "</think>" not in text: # reached the limit: the reasoning is not logged
            total += THINK_CAP
        else:
            total += len(tokenizer.encode(text)) + OVERHEAD[kind]
    loop = "LOOP DETECTED" in open(path, encoding="utf-8").read()
    return total, len(turns), loop

tokens = {} # model -> level -> list of (tokens, steps, loop), one per game
for model, (folder, _, log_seeds) in configurations.items():
    if len(log_seeds) == 0:
        continue
    tokens[model] = {}
    for level in run_levels:
        paths = [f"{folder}/logs/{game}/{model}think_level{level}-seed{seed}.log" for seed in log_seeds]
        tokens[model][level] = [count_tokens(path) for path in paths if os.path.exists(path)]
    print(f"{model}-think: {sum(len(games) for games in tokens[model].values())} logs")

# Comparison
For every level: average normalized score, average generated tokens per game and per step (in thousands), and the games stopped because of a loop (only in 7e).
Stopping the loops makes the games of er4-think shorter, so they generate fewer tokens: tokens per step does not depend on it.

In [ ]:
def tokens_per_game(model, level):
    return np.array([n for n, steps, loop in tokens[model][level]], dtype=float)

def tokens_per_step(model, level):
    return np.array([n / steps for n, steps, loop in tokens[model][level]], dtype=float)

for model in configurations:
    print(f"{model}-think")
    print("level   score   ktokens/game   tokens/step   loops")
    for level in run_levels:
        line = f"{level:>5}   {np.mean(scores[model][level]):.2f}"
        if model in tokens:
            loops = sum(loop for _, _, loop in tokens[model][level])
            line += f"   {np.mean(tokens_per_game(model, level)) / 1000:>12.1f}   {np.mean(tokens_per_step(model, level)):>11.0f}   {loops:>5}"
        print(line)
    if model in tokens:
        all_games = np.concatenate([tokens_per_game(model, level) for level in run_levels])
        all_steps = np.concatenate([tokens_per_step(model, level) for level in run_levels])
        print(f"  all   {np.mean(np.concatenate(list(scores[model].values()))):.2f}   {np.mean(all_games) / 1000:>12.1f}   {np.mean(all_steps):>11.0f}")
    print()

# Graph
Average per level with its 90% bootstrap confidence interval: normalized score, thousands of generated tokens per game, generated tokens per step.
The graph is also saved as a PDF in `Experiment 7e/graphs`.

In [ ]:
# same colors, line styles and markers of the graphs of experiments 6e/7e/8
styles = {"0":   dict(color="#e34948", linestyle=":",  marker="v"),
          "e1":  dict(color="#008300", linestyle="--", marker="s"),
          "r4":  dict(color="#2a78d6", linestyle="-.", marker="o"),
          "er4": dict(color="#4a3aa7", linestyle="-",  marker="D")}
labels = {"0": "0-think", "e1": "e1-think", "r4": "r4-think", "er4": "er4-think (ephemeral)"}

def plot_metric(ax, metric, models, ylabel, title, ylim=None):
    for model in models:
        x, y, low, high = [], [], [], []
        for level in run_levels:
            values = metric(model, level)
            if len(values) < 2:
                continue
            ci = bootstrap((values,), np.mean, method="basic", n_resamples=1000, confidence_level=0.9).confidence_interval
            x.append(level)
            y.append(np.mean(values))
            low.append(ci.low if np.isfinite(ci.low) else y[-1])
            high.append(ci.high if np.isfinite(ci.high) else y[-1])
        if ylim is not None: # a score is between 0 and 1
            low, high = np.clip(low, *ylim), np.clip(high, *ylim)
        ax.fill_between(x, low, high, color=styles[model]["color"], alpha=0.12, linewidth=0)
        ax.plot(x, y, label=labels[model], linewidth=2, markersize=7, **styles[model])
    ax.set_title(title, weight="bold")
    ax.set_xlabel("level")
    ax.set_ylabel(ylabel)
    ax.set_xticks(list(run_levels))
    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_metric(axes[0], lambda model, level: scores[model][level], list(configurations),
            "average normalized final score", "Score (seeds 1-10)", ylim=(0, 1))
plot_metric(axes[1], lambda model, level: tokens_per_game(model, level) / 1000, list(tokens),
            "thousands of generated tokens per game", "Generated tokens per game")
plot_metric(axes[2], tokens_per_step, list(tokens),
            "generated tokens per step", "Generated tokens per step")
fig.suptitle(f"{game.capitalize()} with CoT: experiment 7 vs ephemeral 4-think (7e)")
fig.tight_layout()

os.makedirs("./Experiment 7e/graphs", exist_ok=True)
fig.savefig(f"./Experiment 7e/graphs/tokens_and_scores_{game}.pdf", bbox_inches="tight")
plt.show()